
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Offline vs. Online Evaluation Strategies
#### Duration: 20 minutes

## Overview

Effective agent evaluation requires both offline validation using curated datasets and online monitoring of production performance. This lecture explores the complementary roles of offline and online evaluation, their respective strengths and limitations, and how to create feedback loops between them.

We'll examine when to use each approach, understand their implementation patterns, and learn how to build evaluation systems that evolve with real-world usage while maintaining rigorous quality standards.

## Learning Objectives

By the end of this lecture, you will be able to:
- Identify when to use offline versus online evaluation strategies
- Understand the strengths and limitations of each approach
- Describe how offline and online evaluation complement each other
- Explain how to create feedback loops between production and evaluation
- Recognize best practices for comprehensive evaluation strategies

## A. Evaluation Strategy Overview


### A1. Two Complementary Approaches

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 570" width="100%" role="img" aria-label="Offline and online evaluation side by side. Offline: curate a dataset, run the agent, score responses, and gate the release. Online: trace production requests, score a sample automatically, collect user feedback, and monitor quality. The version that passed offline ships to production, and production failures become new offline test cases." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="oo-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="oo-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="600" height="520" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#oo-sh)"/>
<rect x="62" y="34" width="556" height="5" rx="2.5" fill="#2272B4"/>
<rect x="62" y="53" width="270" height="26" rx="13.0" fill="#2272B4"/><text x="197" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">OFFLINE · BEFORE DEPLOYMENT</text>
<text x="62" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Curated datasets</text>
<rect x="62" y="150" width="556" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<circle cx="92" cy="181" r="15" fill="#2272B4"/>
<text x="92" y="187" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">1</text>
<text x="120" y="188" font-size="19" font-weight="500" fill="#0b2026" text-anchor="start" xml:space="preserve">Curate a dataset of use cases and edge cases</text>
<rect x="62" y="224" width="556" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<circle cx="92" cy="255" r="15" fill="#2272B4"/>
<text x="92" y="261" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">2</text>
<text x="120" y="262" font-size="19" font-weight="500" fill="#0b2026" text-anchor="start" xml:space="preserve">Run the agent on every row</text>
<rect x="62" y="298" width="556" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<circle cx="92" cy="329" r="15" fill="#2272B4"/>
<text x="92" y="335" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">3</text>
<text x="120" y="336" font-size="19" font-weight="500" fill="#0b2026" text-anchor="start" xml:space="preserve">Score each response with judges</text>
<rect x="62" y="372" width="556" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<circle cx="92" cy="403" r="15" fill="#2272B4"/>
<text x="92" y="409" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">4</text>
<text x="120" y="410" font-size="19" font-weight="500" fill="#0b2026" text-anchor="start" xml:space="preserve">Gate the release on the results</text>
<text x="62" y="510" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="start" xml:space="preserve">Controlled and reproducible</text>
<rect x="960" y="20" width="600" height="520" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#oo-sh)"/>
<rect x="982" y="34" width="556" height="5" rx="2.5" fill="#00A972"/>
<rect x="982" y="53" width="224" height="26" rx="13.0" fill="#00A972"/><text x="1094" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">ONLINE · IN PRODUCTION</text>
<text x="982" y="112" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Real user traffic</text>
<rect x="982" y="150" width="556" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<circle cx="1012" cy="181" r="15" fill="#00A972"/>
<text x="1012" y="187" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">1</text>
<text x="1040" y="188" font-size="19" font-weight="500" fill="#0b2026" text-anchor="start" xml:space="preserve">Trace every production request</text>
<rect x="982" y="224" width="556" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<circle cx="1012" cy="255" r="15" fill="#00A972"/>
<text x="1012" y="261" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">2</text>
<text x="1040" y="262" font-size="19" font-weight="500" fill="#0b2026" text-anchor="start" xml:space="preserve">Score a sample of traces automatically</text>
<rect x="982" y="298" width="556" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<circle cx="1012" cy="329" r="15" fill="#00A972"/>
<text x="1012" y="335" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">3</text>
<text x="1040" y="336" font-size="19" font-weight="500" fill="#0b2026" text-anchor="start" xml:space="preserve">Collect user feedback on answers</text>
<rect x="982" y="372" width="556" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
<circle cx="1012" cy="403" r="15" fill="#00A972"/>
<text x="1012" y="409" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">4</text>
<text x="1040" y="410" font-size="19" font-weight="500" fill="#0b2026" text-anchor="start" xml:space="preserve">Monitor quality and alert on drops</text>
<text x="982" y="510" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="start" xml:space="preserve">Real-world and continuous</text>
<text x="800" y="196" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Ship the version that passed</text>
<path d="M648,214 H952" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#oo-ar)"/>
<rect x="700" y="266" width="200" height="44" rx="22" fill="#fff" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="800" y="295" font-size="19" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">Feedback loop</text>
<text x="800" y="372" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Failures become new test cases</text>
<path d="M952,390 H648" fill="none" stroke="#618794" stroke-width="2.4" stroke-dasharray="7 6" marker-end="url(#oo-ar)"/>
</svg>
</div>

Agent evaluation requires both controlled validation and real-world monitoring. Offline and online evaluation serve different but complementary purposes in ensuring agent quality throughout the development and deployment lifecycle.

## B. Offline Evaluation: Pre-Deployment Validation


### B1. Offline Evaluation Workflow

Offline evaluation tests your agent using curated datasets before deployment. Controlled, reproducible, and gated.

1. **Curate a dataset** of common use cases and edge cases.
2. **Define expectations**: optional ground truth or guidelines.
3. **Run the agent** to generate responses.
4. **Apply scorers**: run several judges over every response.
5. **Analyze and iterate**: compare results across runs.

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Strengths and limitations</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
<div style="display:flex;gap:16px;flex-wrap:wrap;">
<div style="flex:1;min-width:200px;">
<strong style="color:#00A972;">Strengths</strong>
<ul style="margin:6px 0;"><li>Rigorous validation before users encounter the agent</li><li>A/B testing different configurations</li><li>Baseline metrics for production</li></ul>
</div>
<div style="flex:1;min-width:200px;">
<strong style="color:#FF5F46;">Limitations</strong>
<ul style="margin:6px 0;"><li>May not fully represent real user behavior</li><li>Static datasets become stale</li><li>Cannot capture scale/diversity issues</li></ul>
</div>
</div>
</div>
</details>

Evaluation Datasets (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/eval-datasets/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/concepts/eval-datasets" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/concepts/eval-datasets/" target="_blank">GCP</a>)

## C. Online Evaluation: Production Monitoring


### C1. Online Evaluation Workflow

Online evaluation analyzes agent performance using real user interactions: capturing actual usage patterns, unexpected inputs, and diverse user populations.

1. **Deploy with tracing**, so every production request is traced. In this course, the Airbnb assistant app logs a trace for each request to its MLflow experiment.
2. **Capture traces** in an MLflow experiment.
3. **Score automatically** by registering scorers and starting them with `.register()` and `.start()`.
4. **Alert and collect feedback** to monitor quality.
5. **Augment the dataset** with production examples.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 12px 16px; border-radius: 4px; margin: 12px 0;">
<strong style="color: #0d47a1; font-size: 13px;">Supported scorer types:</strong>
<span style="font-size: 13px; color: #333;"> Built-in judges, <code>make_judge()</code>, <code>Guidelines</code>, and <code>@scorer</code> decorated functions all support production monitoring via <code>.register()</code> + <code>.start()</code>. Only direct <code>Scorer</code> subclasses and third-party scorers are excluded.</span>
</div>

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Production monitoring capabilities</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
<ul style="margin: 8px 0;">
<li><strong>Auto scoring</strong>: LLM judges run on production traces</li>
<li><strong>Sampling</strong>: configurable sample rates</li>
<li><strong>Alerting</strong>: alerts when quality drops below a threshold</li>
<li><strong>Trace inspection</strong>: deep dives into failures</li>
</ul>
</div>
</details>

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Strengths and limitations</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
<div style="display:flex;gap:16px;flex-wrap:wrap;">
<div style="flex:1;min-width:200px;">
<strong style="color:#00A972;">Strengths</strong>
<ul style="margin:6px 0;"><li>Reflects actual user experience</li><li>Detects unanticipated issues</li><li>Data for continuous improvement</li></ul>
</div>
<div style="flex:1;min-width:200px;">
<strong style="color:#FF5F46;">Limitations</strong>
<ul style="margin:6px 0;"><li>Users may hit failures first</li><li>Harder to attribute root causes</li><li>Requires production infrastructure</li></ul>
</div>
</div>
</div>
</details>

Production Monitoring (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/" target="_blank">GCP</a>) &nbsp;|&nbsp; Inference Tables (<a href="https://docs.databricks.com/aws/en/ai-gateway/inference-tables/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-gateway/inference-tables" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-gateway/inference-tables/" target="_blank">GCP</a>)

## D. Complementary Strategies

### D1. When to Use Each Approach

Effective agent evaluation combines both. Click each to see when it applies:

<div style="max-width:1000px;margin:0 auto;font-family:sans-serif;">

<style>
.oo-step-row { display:flex; gap:12px; margin-bottom:0; flex-wrap:wrap; }
.oo-step-box { position:relative; flex:1; min-width:280px; background:#fff; border:1.5px solid #C6DCEA; border-radius:14px; padding:16px 14px 14px; text-align:center; cursor:pointer; user-select:none; box-shadow:0 2px 8px rgba(27,49,57,.07); transition:transform 0.12s, border-color 0.12s; }
.oo-step-box::before { content:""; display:block; height:5px; border-radius:5px; background:var(--oc); margin:-4px 0 12px; }
.oo-step-box:hover { transform:translateY(-2px); }
.oo-step-box.active { border-color:var(--oc); }
.oo-step-title { display:block; font-size:19px; font-weight:700; color:#0b2026; line-height:1.3; pointer-events:none; }
.oo-step-sub { display:block; font-size:16px; color:#618794; margin-top:4px; pointer-events:none; }
.oo-detail-wrap { overflow:hidden; max-height:0; opacity:0; transition:max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top:0; }
.oo-detail-wrap.open { max-height:800px; opacity:1; margin-top:12px; }
.oo-detail-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:14px; padding:16px 20px 20px; box-shadow:0 2px 8px rgba(27,49,57,.07); font-size:17px; line-height:1.6; }
</style>

<div style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:14px;padding:18px;">

<div class="oo-step-row">
  <span class="oo-step-box active" data-id="0" onclick="selectOoStep(0)" style="--oc:#2272B4;">
    <span class="oo-step-title">Offline Evaluation</span>
    <span class="oo-step-sub">Pre-deployment validation</span>
  </span>
  <span class="oo-step-box" data-id="1" onclick="selectOoStep(1)" style="--oc:#00A972;">
    <span class="oo-step-title">Online Evaluation</span>
    <span class="oo-step-sub">Production monitoring</span>
  </span>
</div>

<div class="oo-detail-wrap" id="oo-detail-wrap">
  <div class="oo-detail-card" id="oo-detail-card"></div>
</div>

</div>
</div>

<script>
var OO_TYPES = [
  {
    title: "Offline Evaluation",
    color: "#2272B4",
    content: '<ul style="margin:8px 0;"><li><strong>Quality gates</strong> - Pre-deployment validation before users see the agent</li><li><strong>A/B testing</strong> - Compare alternative agent implementations side-by-side</li><li><strong>Hypothesis testing</strong> - Test specific behaviors (e.g., "does the new prompt reduce hallucinations?")</li><li><strong>Rapid iteration</strong> - Fast feedback during development cycles</li></ul>'
  },
  {
    title: "Online Evaluation",
    color: "#00A972",
    content: '<ul style="margin:8px 0;"><li><strong>Generalization check</strong> - Validate that offline results hold in production</li><li><strong>Drift detection</strong> - Catch performance degradation over time</li><li><strong>User insight</strong> - Understand real needs, pain points, and edge cases</li><li><strong>Dataset mining</strong> - Build evaluation datasets from actual usage patterns</li></ul>'
  }
];

var ooCurrent = null;

function selectOoStep(id) {
  var wrap = document.getElementById('oo-detail-wrap');
  var card = document.getElementById('oo-detail-card');
  var o = OO_TYPES[id];

  document.querySelectorAll('.oo-step-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id, 10) === id);
  });

  if (ooCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.oo-step-box').forEach(function(b) { b.classList.remove('active'); });
    ooCurrent = null;
    return;
  }

  ooCurrent = id;
  card.innerHTML =
    '<div style="height:5px;border-radius:5px;margin:-4px 0 12px;background:' + o.color + ';"></div>' +
    '<div style="font-size:21px;font-weight:700;margin-bottom:10px;color:#0b2026;">' + o.title + '</div>'
    + o.content;

  wrap.classList.add('open');
}

selectOoStep(0);
</script>


### D2. Creating a Feedback Loop

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 802" width="100%" role="img" aria-label="Swimlane of the production feedback loop across users, MLflow, and the developer: a user asks the Airbnb assistant, the app logs a trace, the user rates the answer, user_feedback is saved on the trace, the developer finds the failures with search_traces for the evaluation dataset, fixes the agent and re-runs evaluate, and registers a new version as @champion, which the app then serves." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="sl-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="sl-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<clipPath id="sl-lane0"><rect x="40" y="90" width="1520" height="220" rx="14"/></clipPath>
<g clip-path="url(#sl-lane0)"><rect x="40" y="90" width="1520" height="220" fill="#F9F7F4"/><rect x="40" y="90" width="8" height="220" fill="#2272B4"/></g>
<rect x="40" y="90" width="1520" height="220" rx="14" fill="none" stroke="#EEEDE9"/>
<text x="66" y="192" font-size="15" font-weight="800" fill="#2272B4" text-anchor="start" letter-spacing="1.2" xml:space="preserve">USERS</text>
<text x="66" y="216" font-size="15" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Airbnb</text>
<text x="66" y="236" font-size="15" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">assistant app</text>
<clipPath id="sl-lane1"><rect x="40" y="324" width="1520" height="220" rx="14"/></clipPath>
<g clip-path="url(#sl-lane1)"><rect x="40" y="324" width="1520" height="220" fill="#fff"/><rect x="40" y="324" width="8" height="220" fill="#FF5F46"/></g>
<rect x="40" y="324" width="1520" height="220" rx="14" fill="none" stroke="#EEEDE9"/>
<text x="66" y="426" font-size="15" font-weight="800" fill="#FF5F46" text-anchor="start" letter-spacing="1.2" xml:space="preserve">MLFLOW</text>
<text x="66" y="450" font-size="15" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Traces and</text>
<text x="66" y="470" font-size="15" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">assessments</text>
<clipPath id="sl-lane2"><rect x="40" y="558" width="1520" height="220" rx="14"/></clipPath>
<g clip-path="url(#sl-lane2)"><rect x="40" y="558" width="1520" height="220" fill="#F9F7F4"/><rect x="40" y="558" width="8" height="220" fill="#1B5162"/></g>
<rect x="40" y="558" width="1520" height="220" rx="14" fill="none" stroke="#EEEDE9"/>
<text x="66" y="660" font-size="15" font-weight="800" fill="#1B5162" text-anchor="start" letter-spacing="1.2" xml:space="preserve">DEVELOPER</text>
<text x="66" y="684" font-size="15" font-weight="500" fill="#5A6F77" text-anchor="start" xml:space="preserve">Notebooks</text>
<rect x="180" y="112" width="172" height="176" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<circle cx="210" cy="144" r="15" fill="#2272B4"/>
<text x="210" y="150" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">1</text>
<text x="234" y="151" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Ask</text>
<text x="196" y="186" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">A user asks the</text>
<text x="196" y="211" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Airbnb</text>
<text x="196" y="236" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">assistant.</text>
<rect x="381" y="346" width="172" height="176" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<circle cx="411" cy="378" r="15" fill="#FF5F46"/>
<text x="411" y="384" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">2</text>
<text x="435" y="385" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Trace</text>
<text x="397" y="420" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">The app logs a</text>
<text x="397" y="445" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">trace to its</text>
<text x="397" y="470" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">experiment.</text>
<rect x="582" y="112" width="172" height="176" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<circle cx="612" cy="144" r="15" fill="#2272B4"/>
<text x="612" y="150" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">3</text>
<text x="636" y="151" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Rate</text>
<text x="598" y="186" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">The user picks</text>
<text x="598" y="211" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Yes or No, plus</text>
<text x="598" y="236" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">a comment.</text>
<rect x="783" y="346" width="172" height="176" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<circle cx="813" cy="378" r="15" fill="#FF5F46"/>
<text x="813" y="384" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">4</text>
<text x="837" y="385" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Assess</text>
<text x="799" y="420" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">user_feedback is</text>
<text x="799" y="445" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">saved on the</text>
<text x="799" y="470" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">trace.</text>
<rect x="984" y="580" width="172" height="176" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<circle cx="1014" cy="612" r="15" fill="#1B5162"/>
<text x="1014" y="618" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">5</text>
<text x="1038" y="619" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Curate</text>
<text x="1000" y="654" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">search_traces()</text>
<text x="1000" y="679" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">finds the No</text>
<text x="1000" y="704" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">answers for the</text>
<text x="1000" y="729" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">eval dataset.</text>
<rect x="1185" y="580" width="172" height="176" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<circle cx="1215" cy="612" r="15" fill="#1B5162"/>
<text x="1215" y="618" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">6</text>
<text x="1239" y="619" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Fix</text>
<text x="1201" y="654" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Change the agent</text>
<text x="1201" y="679" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">and re-run</text>
<text x="1201" y="704" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">evaluate().</text>
<rect x="1386" y="346" width="172" height="176" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#sl-sh)"/>
<circle cx="1416" cy="378" r="15" fill="#FF5F46"/>
<text x="1416" y="384" font-size="15" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">7</text>
<text x="1440" y="385" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Promote</text>
<text x="1402" y="420" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Register a</text>
<text x="1402" y="445" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">version and move</text>
<text x="1402" y="470" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">@champion.</text>
<path d="M266.0,291 V420.0 Q266.0,434.0 280.0,434.0 H375" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<path d="M467.0,343 V214.0 Q467.0,200.0 481.0,200.0 H576" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<path d="M668.0,291 V420.0 Q668.0,434.0 682.0,434.0 H777" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<path d="M869.0,525 V654.0 Q869.0,668.0 883.0,668.0 H978" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<path d="M1159,668.0 H1179" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<path d="M1271.0,577 V448.0 Q1271.0,434.0 1285.0,434.0 H1380" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#sl-ar)"/>
<path d="M1472.0,343 V60 H266.0 V106" fill="none" stroke="#618794" stroke-width="2.4" stroke-dasharray="7 6" marker-end="url(#sl-ar)"/>
<rect x="669.0" y="38" width="400" height="44" rx="22" fill="#fff" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="869" y="67" font-size="18" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">The app serves the new champion</text>
</svg>
</div>

This cycle ensures your evaluation evolves with your understanding of real-world usage while maintaining rigorous pre-deployment validation.

Human Feedback (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/human-feedback/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/human-feedback/" target="_blank">GCP</a>)

## E. Integration with Unity Catalog


### E1. Governance for Evaluation

Unity Catalog provides the governance layer for your evaluation workflow: traceability, access control, and lineage tracking across agents, datasets, and traces.

- **Agent registration**: dependencies, access controls, and aliases such as `@champion`
- **Dataset storage**: versioned, governed evaluation datasets
- **Trace storage**: SQL-queryable traces you can share across teams
- **Lineage**: from datasets to metrics to agent versions

<details style="margin: 14px 0; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Details</summary>
<div style="padding: 0 0 18px 0; font-size: 16px; color: #5A6F77; line-height: 1.6;">
<strong>Agent registration:</strong> All dependencies (vector indexes, UC functions, endpoints) are tracked. Aliases like <code>@champion</code> enable clean promotion workflows.<br/><br/>
<strong>Dataset storage:</strong> Version datasets alongside agent versions, apply access controls, enable SQL-based analysis and collaborative curation.<br/><br/>
<strong>Trace storage:</strong> Query traces using SQL, join with evaluation results, apply governance policies, and enable cross-team sharing.
</div>
</details>

MLflow Tracing (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/" target="_blank">GCP</a>) &nbsp;|&nbsp; Evaluation Datasets (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/eval-datasets/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/eval-monitor/concepts/eval-datasets" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/eval-monitor/concepts/eval-datasets/" target="_blank">GCP</a>)


### E2. Storing Traces in Unity Catalog Tables

By default, MLflow stores traces in the MLflow control plane service. For production workloads, you can instead store OpenTelemetry traces directly in Unity Catalog managed tables. This enables SQL-based analytics over traces, UC-level access control, and long-term retention.

<div class="mlt">
<style>
  .mlt{background:#ffffff;max-width:1600px;margin:0 auto;padding:24px 20px 40px;}
  .mlt svg{display:block;width:100%;height:auto;}
  .mlt svg text{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif;fill:#0b2026;}
  .mlt .h   {font-weight:800;}
  .mlt .b   {font-weight:700;}
  .mlt .m   {fill:#618794;font-weight:500;}
  .mlt .mono,.mlt .mono text{white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace;font-variant-ligatures:none;font-feature-settings:"liga" 0,"calt" 0;}
  .mlt .pfx {fill:#FF5F46;}
</style>
<svg viewBox="0 0 1600 680" role="img" aria-label="MLflow traces registered with Unity Catalog. The Airbnb assistant app, the registered agent, its tool calls, evaluation runs, autologged frameworks, or custom code are instrumented by MLflow Tracing, which captures inputs and outputs, latency, token usage, tool calls, errors and span hierarchy. Binding the experiment to UnityCatalog with table_prefix airbnb writes traces into the prod.agents schema as four managed tables: airbnb_otel_spans, airbnb_otel_logs, airbnb_otel_annotations and airbnb_otel_metrics, plus two SQL views built over them: airbnb_trace_unified and airbnb_trace_metadata. The assets are governed by Unity Catalog and queried with Databricks SQL or mlflow.search_traces.">
  <defs>
    <!-- repo glyphs: product-icons/tableIcon.svg and product-icons/TableViewIcon.svg -->
    <symbol id="ic-table" viewBox="0 0 16 16">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M1 1.75C1 1.33579 1.33579 1 1.75 1H14.25C14.6642 1 15 1.33579 15 1.75V14.25C15 14.6642 14.6642 15 14.25 15H1.75C1.33579 15 1 14.6642 1 14.25V1.75ZM2.5 2.5V5.5H13.5V2.5H2.5ZM2.5 13.5V7H5V13.5H2.5ZM6.5 13.5H9.5V7H6.5V13.5ZM11 7V13.5H13.5V7H11Z" fill="#0B2026"/>
    </symbol>
    <symbol id="ic-view" viewBox="0 0 16 16">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M1.75 1C1.33579 1 1 1.33579 1 1.75V14.25C1 14.6642 1.33579 15 1.75 15H4V13.5H2.5V7H5V9H6.5V7H9.5V9H11V7H13.5V9H15V1.75C15 1.33579 14.6642 1 14.25 1H1.75ZM13.5 5.5V2.5H2.5V5.5H13.5Z" fill="#FF3621"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M11.75 10C11.5511 10 11.3603 10.079 11.2197 10.2197C11.1398 10.2995 11.0798 10.3955 11.0429 10.5H9.95735C9.85439 10.2087 9.57658 10 9.25003 10L5.75 10C5.55108 10 5.36031 10.079 5.21966 10.2197C5.07901 10.3603 4.99999 10.5511 5 10.75L5.00003 12.5C5.00003 13.8807 6.11932 15 7.50003 15C8.88074 15 10 13.8807 10 12.5V12H11L11 12.5C11 13.8807 12.1193 15 13.5 15C14.8807 15 16 13.8807 16 12.5V10.7506C16 10.3364 15.6642 10 15.25 10L11.75 10ZM12.5 12.5L12.5 11.5L14.5 11.5V12.5C14.5 13.0523 14.0523 13.5 13.5 13.5C12.9477 13.5 12.5 13.0523 12.5 12.5ZM6.50002 11.5L6.50003 12.5C6.50003 13.0523 6.94774 13.5 7.50003 13.5C8.05231 13.5 8.50003 13.0523 8.50003 12.5V11.5001L6.50002 11.5Z" fill="#FF3621"/>
    </symbol>
    <marker id="ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
    <marker id="ar-big" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6" markerHeight="6" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#1B5162"/></marker>
    <filter id="sh" x="-10%" y="-10%" width="120%" height="125%"><feDropShadow dx="0" dy="4" stdDeviation="9" flood-color="#1B3139" flood-opacity="0.10"/></filter>
  </defs>
  <!-- ================= 1 · GenAI app ================= -->
  <rect x="40" y="20" width="320" height="620" rx="18" fill="#ffffff" stroke="#DCE0E2" stroke-width="1.5" filter="url(#sh)"/>
  <text class="m b" x="68" y="60" font-size="16" letter-spacing="1.6">1 · YOUR AGENT</text>
  <text class="h" x="68" y="90" font-size="25">Anything that traces</text>
  <text class="m" x="68" y="116" font-size="17">the course agent, or any framework</text>
  <g>
    <rect x="62" y="142" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="168" font-size="19">Airbnb assistant app</text>
    <text class="mono m" x="82" y="190" font-size="15">@invoke / @stream</text>
    <rect x="62" y="216" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="242" font-size="19">Registered agent</text>
    <text class="mono m" x="82" y="264" font-size="15">ResponsesAgent.predict()</text>
    <rect x="62" y="290" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="316" font-size="19">Tool calls</text>
    <text class="mono m" x="82" y="338" font-size="15">mlflow.start_span()</text>
    <rect x="62" y="364" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="390" font-size="19">Evaluation runs</text>
    <text class="mono m" x="82" y="412" font-size="15">mlflow.genai.evaluate()</text>
    <rect x="62" y="438" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="464" font-size="19">OpenAI, LangChain, DSPy</text>
    <text class="mono m" x="82" y="486" font-size="15">mlflow.&lt;flavor&gt;.autolog()</text>
    <rect x="62" y="512" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="538" font-size="19">Custom code</text>
    <text class="mono m" x="82" y="560" font-size="15">@mlflow.trace</text>
  </g>
  <text class="m" x="200" y="612" font-size="16" text-anchor="middle">no changes to app logic</text>
  <!-- arrow 1 -> 2 -->
  <path d="M362,330 L414,330" fill="none" stroke="#618794" stroke-width="2.5" marker-end="url(#ar)"/>
  <!-- ================= 2 · MLflow Tracing ================= -->
  <rect x="420" y="20" width="320" height="620" rx="18" fill="#ffffff" stroke="#DCE0E2" stroke-width="1.5" filter="url(#sh)"/>
  <text class="m b" x="448" y="60" font-size="16" letter-spacing="1.6">2 · MLFLOW TRACING</text>
  <text class="h" x="448" y="90" font-size="25">Each trace captures</text>
  <g font-size="19">
    <circle cx="456" cy="130" r="4" fill="#4299E0"/><text x="470" y="136">Inputs &amp; outputs</text>
    <circle cx="456" cy="162" r="4" fill="#4299E0"/><text x="470" y="168">Latency per step</text>
    <circle cx="456" cy="194" r="4" fill="#4299E0"/><text x="470" y="200">Token usage</text>
    <circle cx="456" cy="226" r="4" fill="#4299E0"/><text x="470" y="232">Tool / function calls</text>
    <circle cx="456" cy="258" r="4" fill="#4299E0"/><text x="470" y="264">Errors &amp; retries</text>
    <circle cx="456" cy="290" r="4" fill="#4299E0"/><text x="470" y="296">Span hierarchy</text>
  </g>
  <text class="m b" x="448" y="350" font-size="16" letter-spacing="1.6">BIND TO UNITY CATALOG</text>
  <rect x="440" y="364" width="280" height="168" rx="10" fill="#303F47"/>
  <g class="mono" font-size="14">
    <text x="456" y="392" style="fill:#C4CCD6">mlflow.set_experiment(</text>
    <text x="456" y="414" style="fill:#C4CCD6">  experiment_name="…",</text>
    <text x="456" y="436" style="fill:#C4CCD6">  trace_location=UnityCatalog(</text>
    <text x="456" y="458" style="fill:#C4CCD6">    catalog_name=<tspan style="fill:#FFCC66">"prod"</tspan>,</text>
    <text x="456" y="480" style="fill:#C4CCD6">    schema_name=<tspan style="fill:#FFCC66">"agents"</tspan>,</text>
    <text x="456" y="502" style="fill:#C4CCD6">    table_prefix=<tspan style="fill:#FF8774">"airbnb"</tspan>))</text>
  </g>
  <text class="m" x="448" y="570" font-size="16">Needs a SQL warehouse:</text>
  <text class="mono m" x="448" y="592" font-size="15">MLFLOW_TRACING_SQL_WAREHOUSE_ID</text>
  <!-- arrow 2 -> 3 -->
  <path d="M742,330 L794,330" fill="none" stroke="#618794" stroke-width="2.5" marker-end="url(#ar)"/>
  <!-- ================= 3 · Unity Catalog ================= -->
  <rect x="800" y="20" width="760" height="620" rx="18" fill="#ffffff" stroke="#1B5162" stroke-width="1.8" filter="url(#sh)"/>
  <path d="M818,20 H1542 A18,18 0 0 1 1560,38 V76 H800 V38 A18,18 0 0 1 818,20 Z" fill="#1B5162"/>
  <text x="826" y="56" font-size="25" font-weight="800" style="fill:#ffffff">3 · Unity Catalog</text>
  <text class="mono" x="1534" y="55" font-size="16" text-anchor="end" style="fill:#ffffff">prod.agents.<tspan style="fill:#FF8774">airbnb</tspan>_*</text>
  <!-- tables section -->
  <use href="#ic-table" x="826" y="96" width="20" height="20"/>
  <text class="b" x="854" y="113" font-size="19">4 managed tables</text>
  <text class="m" x="1012" y="113" font-size="17">raw OpenTelemetry data, written as traces arrive</text>
  <g>
    <!-- spans -->
    <path d="M822,130 H1164 A10,10 0 0 1 1174,140 V212 A10,10 0 0 1 1164,222 H822 Z" fill="#F9F7F4" stroke="#EEEDE9"/>
    <rect x="822" y="130" width="5" height="92" fill="#1B5162"/>
    <use href="#ic-table" x="842" y="146" width="22" height="22"/>
    <text class="mono b" x="874" y="164" font-size="18"><tspan class="pfx">airbnb</tspan>_otel_spans</text>
    <text class="m" x="842" y="192" font-size="17">One row per span: the call tree with</text>
    <text class="m" x="842" y="211" font-size="17">timing, inputs and outputs</text>
    <!-- logs -->
    <path d="M1186,130 H1528 A10,10 0 0 1 1538,140 V212 A10,10 0 0 1 1528,222 H1186 Z" fill="#F9F7F4" stroke="#EEEDE9"/>
    <rect x="1186" y="130" width="5" height="92" fill="#1B5162"/>
    <use href="#ic-table" x="1206" y="146" width="22" height="22"/>
    <text class="mono b" x="1238" y="164" font-size="18"><tspan class="pfx">airbnb</tspan>_otel_logs</text>
    <text class="m" x="1206" y="192" font-size="17">OpenTelemetry log records emitted</text>
    <text class="m" x="1206" y="211" font-size="17">while the app runs</text>
    <!-- annotations -->
    <path d="M822,234 H1164 A10,10 0 0 1 1174,244 V316 A10,10 0 0 1 1164,326 H822 Z" fill="#F9F7F4" stroke="#EEEDE9"/>
    <rect x="822" y="234" width="5" height="92" fill="#1B5162"/>
    <use href="#ic-table" x="842" y="250" width="22" height="22"/>
    <text class="mono b" x="874" y="268" font-size="18"><tspan class="pfx">airbnb</tspan>_otel_annotations</text>
    <text class="m" x="842" y="296" font-size="17">Assessments, feedback and tags</text>
    <text class="m" x="842" y="315" font-size="17">attached to traces</text>
    <!-- metrics -->
    <path d="M1186,234 H1528 A10,10 0 0 1 1538,244 V316 A10,10 0 0 1 1528,326 H1186 Z" fill="#F9F7F4" stroke="#EEEDE9"/>
    <rect x="1186" y="234" width="5" height="92" fill="#1B5162"/>
    <use href="#ic-table" x="1206" y="250" width="22" height="22"/>
    <text class="mono b" x="1238" y="268" font-size="18"><tspan class="pfx">airbnb</tspan>_otel_metrics</text>
    <text class="m" x="1206" y="296" font-size="17">OpenTelemetry metric data points</text>
    <text class="m" x="1206" y="315" font-size="17">recorded alongside traces</text>
  </g>
  <!-- bus: tables -> views -->
  <g fill="none" stroke="#1B5162" stroke-width="2.2" stroke-linecap="round">
    <path d="M998,326 L998,364"/>
    <path d="M1362,326 L1362,364"/>
    <path d="M998,364 L1362,364"/>
    <path d="M998,364 L998,406" marker-end="url(#ar-big)"/>
    <path d="M1362,364 L1362,406" marker-end="url(#ar-big)"/>
  </g>
  <rect x="1056" y="350" width="252" height="28" rx="14" fill="#ffffff" stroke="#1B5162" stroke-width="1.5"/>
  <use href="#ic-view" x="1068" y="356" width="16" height="16"/>
  <text class="b" x="1196" y="370" font-size="16" text-anchor="middle">2 SQL views over the tables</text>
  <!-- views -->
  <g>
    <path d="M822,412 H1164 A10,10 0 0 1 1174,422 V526 A10,10 0 0 1 1164,536 H822 Z" fill="#FFF6F4" stroke="#FABFBA"/>
    <rect x="822" y="412" width="5" height="124" fill="#FF5F46"/>
    <use href="#ic-view" x="842" y="428" width="22" height="22"/>
    <text class="mono b" x="874" y="446" font-size="18"><tspan class="pfx">airbnb</tspan>_trace_unified</text>
    <text class="m" x="842" y="476" font-size="17">One row per trace: spans, metadata</text>
    <text class="m" x="842" y="495" font-size="17">and assessments together</text>
    <rect x="842" y="506" width="120" height="24" rx="12" fill="#FF5F46"/>
    <text x="902" y="523" font-size="14" font-weight="700" text-anchor="middle" style="fill:#ffffff">FULL DETAIL</text>
    <path d="M1186,412 H1528 A10,10 0 0 1 1538,422 V526 A10,10 0 0 1 1528,536 H1186 Z" fill="#FFF6F4" stroke="#FABFBA"/>
    <rect x="1186" y="412" width="5" height="124" fill="#FF5F46"/>
    <use href="#ic-view" x="1206" y="428" width="22" height="22"/>
    <text class="mono b" x="1238" y="446" font-size="18"><tspan class="pfx">airbnb</tspan>_trace_metadata</text>
    <text class="m" x="1206" y="476" font-size="17">Tags, metadata and assessments only,</text>
    <text class="m" x="1206" y="495" font-size="17">no span payloads</text>
    <rect x="1206" y="506" width="156" height="24" rx="12" fill="#FF5F46"/>
    <text x="1284" y="523" font-size="14" font-weight="700" text-anchor="middle" style="fill:#ffffff">FAST FILTERING</text>
  </g>
  <!-- what you get -->
  <line x1="822" y1="556" x2="1538" y2="556" stroke="#EEEDE9" stroke-width="1.5"/>
  <g font-size="16">
    <rect x="822" y="574" width="236" height="44" rx="22" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text x="940" y="601" text-anchor="middle"><tspan class="b">Govern</tspan><tspan class="m">: grants · lineage</tspan></text>
    <rect x="1070" y="574" width="212" height="44" rx="22" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text x="1176" y="601" text-anchor="middle"><tspan class="b">Query</tspan><tspan class="m">: Databricks SQL</tspan></text>
    <rect x="1294" y="574" width="244" height="44" rx="22" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text x="1416" y="601" text-anchor="middle"><tspan class="b">Read</tspan><tspan class="m">: </tspan><tspan class="mono m" font-size="14">mlflow.search_traces()</tspan></text>
  </g>
</svg>
</div>

Traces flow from your agent through MLflow into UC managed tables, queryable via SQL.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Key benefits over default trace storage:</strong>
<ul style="margin: 8px 0 0 16px; color: #333;">
<li><strong>SQL queryable</strong>: Join traces with evaluation results using Databricks SQL</li>
<li><strong>UC access control</strong>: Govern trace access with catalog/schema permissions instead of experiment ACLs</li>
<li><strong>Long-term retention</strong>: Managed tables for high-volume production trace storage</li>
<li><strong>OTel compatible</strong>: Works with third-party OpenTelemetry clients via OTLP exporters</li>
</ul>
<p style="margin: 10px 0 0 0;">
For setup instructions, see <a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog" style="color: #1976d2;">Store OpenTelemetry Traces in Unity Catalog</a> (<a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/trace-unity-catalog" style="color: #1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/trace-unity-catalog" style="color: #1976d2;">GCP</a>).
</p>
</div>

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">

  <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>

  <img src="https://files.training.databricks.com/binder/prod_main/genai-agent-evaluation-on-databricks-en_us-1.2.0/images/20261116T170304Z/Agent Evaluation on Databricks/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to learn more about connecting offline and online evaluation? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/genai-agent-evaluation-on-databricks-en_us-1.2.0/images/20261116T170304Z/Agent Evaluation on Databricks/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #F9F7F4; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-3-1" style="flex: 1;">How do I set up online evaluation with MLflow to monitor agent quality in production?</span>
    <button onclick="
      var text = document.getElementById('genie-3-1').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
